This notebook creates a gene metadata file that will be added to the .var slot of subsequently created h5ads.

The gene metadata will have the following information:

- gene stable ID
- gene name
- IsMouseTF (mouse transcription factors)
- IsHumanTFOrtholog (genes whose orthologs are TFs in humans)
- IsSourceGene (ligands, signaling molecules, ...)
- IsTargetGene (receptors, ...)
- IsHvgExcludedGene (Kim et al. genes found to be sex-specific, retro-virus-induced, noise-sensitive or immediate early genes as well as ribosomal/mitochondrial genes)
- IsAnnotationExcludedGene (IsHvgExcludedGene + pseudo- and Riken-genes)
- IsPriorityGene (genes prioritized for ABC-like and own annotation)
- IsTopGene (genes receiving extra attention for own annotation due to being recognizable within the field)


IsHVG will be added to .var at later stages


# Setup

Unfortunately, you have to manually download some data as it can neither be downladed with wget, curl or requests.

Put the following files into DATADIR/2b_clean


Save [this](https://guolab.wchscu.cn/AnimalTFDB4_static/download/TF_list_final/Mus_musculus_TF) as Mus_musculus_TF.tsv,

[this](https://guolab.wchscu.cn/AnimalTFDB4_static/download/TF_list_final/Homo_sapiens_TF) as Homo_sapiens_TF.tsv,

[this](https://guolab.wchscu.cn/AnimalTFDB4_static/download/ortholog_to_human_download/Mus_musculus_ortholog_to_human) as Mus_musculus_ortholog_to_human.tsv,

and [this](https://pmc.ncbi.nlm.nih.gov/articles/instance/7534821/bin/NIHMS1629366-supplement-Table_S2.xlsx) as NIHMS1629366-supplement-Table_S2.xlsx

# Imports

In [ ]:
from  pathlib import Path
import re

import pandas as pd
import scanpy as sc
from dotenv import dotenv_values

# Constants & Environment

In [ ]:
# Project-wide constants
CONFIG = dotenv_values("config.env") # assuming kernel located at the root of the directory
DATA_ROOT_DIR = Path(CONFIG['DATADIR'])

# Notebook-specific constants
TEMPLATE_FILE = DATA_ROOT_DIR / "2a_cellbender" / "outs" / "wo_empty_droplets" / "SC1-01_SAMN18754353.h5" # does not matter which dataset we choose, as long as it was processed with our alignment steps using GRCm39
META_DIR = DATA_ROOT_DIR / "2b_clean"

OUTFILE = DATA_ROOT_DIR / "gene_metadata.csv"
GENE_INFO_PATH = "resources/gene_annotations.txt"

# Metadata to be downloaded 
HUMAN_TF_PATH = META_DIR / "Homo_sapiens_TF.tsv"
MOUSE_TF_PATH = META_DIR / "Mus_musculus_TF.tsv"
MOUSE_ORTHOLOG_PATH = META_DIR / "Mus_musculus_ortholog_to_human.tsv"
KIM_EXCLUDE_PATH = META_DIR / "NIHMS1629366-supplement-Table_S2.xlsx"


# Let's get the template

In [22]:
adata = sc.read_10x_h5(TEMPLATE_FILE,)
adata.var_names_make_unique()
meta = adata.var["gene_ids"].reset_index().rename(columns={"index":"gene_name"})

/beegfs/home/mfeldmann/repos/HypoMap3D/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/beegfs/home/mfeldmann/repos/HypoMap3D/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1808: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


### Let's intialize columns

In [23]:
cols = [
    "IsMouseTF", 
    "IsHumanTFOrtholog",
    "IsSourceGene",
    "IsTargetGene",
    "IsHvgExcludedGene",
    "IsAnnotationExcludedGene",
    "IsPriorityGene",
    "IsTopGene",]

meta = pd.concat([meta,
                pd.DataFrame(False, index=range(len(meta)), columns=cols)], axis=1)


In [24]:
meta.head()

,gene_name,gene_ids,IsMouseTF,IsHumanTFOrtholog,IsSourceGene,IsTargetGene,IsHvgExcludedGene,IsAnnotationExcludedGene,IsPriorityGene,IsTopGene
0,Gm38212,ENSMUSG00000104478,False,False,False,False,False,False,False,False
1,Gm7449,ENSMUSG00000104385,False,False,False,False,False,False,False,False
2,Gm28283,ENSMUSG00000101231,False,False,False,False,False,False,False,False
3,Gm37108,ENSMUSG00000102135,False,False,False,False,False,False,False,False
4,Gm37275,ENSMUSG00000103282,False,False,False,False,False,False,False,False


# Combining metadata

In [ ]:
Homo_sapiens_TF = pd.read_csv(HUMAN_TF_PATH, sep="\t",)
Mus_musculus_TF = pd.read_csv(MOUSE_TF_PATH, sep="\t")
Mus_ortholog = pd.read_csv(MOUSE_ORTHOLOG_PATH, sep="\t")
gene_info = pd.read_csv(GENE_INFO_PATH, sep="\t")

kim_genes = pd.read_excel(KIM_EXCLUDE_PATH, engine="openpyxl") # Kim et al

/beegfs/home/mfeldmann/repos/HypoMap3D/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


## Let's start with TFs and ligands/receptors

In [ ]:
meta.loc[meta.gene_ids.isin(Mus_musculus_TF.Ensembl.to_list()), "IsMouseTF"] = True

meta.loc[meta.gene_ids.isin(
    Mus_ortholog.loc[Mus_ortholog["Ortholog ID"].isin(Homo_sapiens_TF.Ensembl.to_list()), "Ensembl ID"].to_list() # all genes that are human TFs
), "IsHumanTFOrtholog"] = True

meta.loc[meta.gene_name.isin(gene_info.loc[gene_info.type == "ligand/source", "gene"].to_list()), "IsSourceGene" ] = True

meta.loc[meta.gene_name.isin(gene_info.loc[gene_info.type == "receptor", "gene"].to_list()), "IsTargetGene" ] = True

## Add top genes

Top Genes are genes we want to preferentially use for cell type annotation as these genes are often used to refer to highly-studied cell populations

In [95]:
top_genes = ["Pomc","Agrp","Pmch","Npy","Npw","Sst","Ghrh","Kiss1","Crh","Oxt","Avp","Vip","Grp","Hcrt","Qrfp",
             "Npvf","Gnrh1","Apln","Gal","Adcyap1","Prlh","Ucn","Tac1","Cck","Pdyn","Gcg","Th","Ddc","Dbh","Chat","Glipr1"]

meta.loc[meta.gene_name.isin(top_genes), "IsTopGene"] = True

## Genes exluded for HVG selection and annotation

For both, HVG selection and cluster annotation, we excluded genes from Kim et al.

In [ ]:
# combine all kim genes into one column and get list

kim_genes_list = [] 
for c in kim_genes.columns:
    kim_genes_list.extend(kim_genes[c].to_list())

kim_genes_list = list(set(kim_genes_list))

print(f"Kim genes are {[c for c in kim_genes.columns]} genes such as")
print(kim_genes_list[:5])


Kim genes are ['Sex-specific', 'IEGs', 'retro-virus-induced', 'noise-sensitive'] genes such as
['Ppp2r1a', 'Zfand5', 'Vapa', 'Ndufb10', 'Ahi1']


### HVG selection

For HVG selection, Kim genes as well as ribosomal or mitochondrial genes were excluded from being candidate HVGs before retrieving 4500 HVGs

In [ ]:
ignore_hvg_genes_regex = r"Xist|Uty|Utx|Fos|Fosb|Jun|Junb|Jund|Malat1|Heph|Rpl|Rps|Cox|rpl|rps|mt-|Sv2" 
ignore_hvg_genes = [g for g in meta.gene_name if re.search(ignore_hvg_genes_regex, g)]

ignore_hvg_genes = set(ignore_hvg_genes) | set(kim_genes_list)

# do not exclude
ignore_hvg_genes -= {"Nfib", "Prpsap2"}
ignore_hvg_genes -= set(gene_info.gene.to_list()) # leave ligands and receptors in


In [ ]:
meta.loc[meta.gene_name.isin(ignore_hvg_genes), "IsHvgExcludedGene"] = True

### Genes to exclude for annotation

For annotation of clusters, we additionally excluded pseudo and Riken genes

In [99]:
ignore_annotation_gene_regex = r"Xist|Uty|Utx|Fos|Fosb|Jun|Junb|Jund|Malat1|Heph|Rpl|Rps|Cox|rpl|rps|mt-|Sv2|ENSMUS|Rik|Gm[0-9]|D5Er|Mir|[0-9]+os|-ps"
ignore_annotation_genes = [g for g in meta.gene_name if re.search(ignore_annotation_gene_regex, g)]

ignore_annotation_genes = set(ignore_annotation_genes) | set(kim_genes_list)

# do not exclude
ignore_annotation_genes -= {"Nfib", "Prpsap2"}
ignore_annotation_genes -= set(gene_info.gene.to_list()) # leave ligands and receptors in

# exclude manually
ignore_annotation_genes |= {"Vgf","D7Ertd443e"}

In [100]:
meta.loc[meta.gene_name.isin(ignore_annotation_genes), "IsAnnotationExcludedGene"] = True

## Flag priority genes

Priority genes are all TFs/ligands/receptor genes but only if they are not excluded for annotation.

In [ ]:
priority_mask = (meta.IsMouseTF | meta.IsHumanTFOrtholog | meta.IsSourceGene | meta.IsTargetGene| meta.IsTopGene) & ~(meta.IsAnnotationExcludedGene)

In [109]:
meta.loc[priority_mask, "IsPriorityGene"] = True

# Save as csv

In [112]:
meta.head()

,gene_name,gene_ids,IsMouseTF,IsHumanTFOrtholog,IsSourceGene,IsTargetGene,IsHvgExcludedGene,IsAnnotationExcludedGene,IsPriorityGene,IsTopGene
0,Gm38212,ENSMUSG00000104478,False,False,False,False,False,True,False,False
1,Gm7449,ENSMUSG00000104385,False,False,False,False,False,True,False,False
2,Gm28283,ENSMUSG00000101231,False,False,False,False,False,True,False,False
3,Gm37108,ENSMUSG00000102135,False,False,False,False,False,True,False,False
4,Gm37275,ENSMUSG00000103282,False,False,False,False,False,True,False,False


In [ ]:
meta.to_csv(OUTFILE, index=False)